<a href="https://colab.research.google.com/github/Haftah01/ECC/blob/main/academic_research_agent_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

# Securely load API Keys from Colab Secrets
for key in ['OPENROUTER_API_KEY', 'EXA_API_KEY', 'SEMANTIC_SCHOLAR_API_KEY', 'TAVILY_API_KEY']:
    try:
        os.environ[key] = userdata.get(key)
        print(f"✅ {key} loaded.")
    except:
        print(f"⚠️ {key} not found in secrets.")

In [ ]:
# 1. Install standard packages including langchain-openai for OpenRouter support
!pip install -q langgraph langchain-nvidia-ai-endpoints langchain-community tavily-python gradio langchain-openai

In [ ]:
import os
import operator
import requests
from typing import Annotated, TypedDict, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

# --- 1. Models (via OpenRouter) ---
reasoning_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model="nvidia/nemotron-3-ultra-550b-a55b:free"
)

writer_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model="nvidia/nemotron-3-ultra-550b-a55b:free"
)

# --- 2. Academic Tools ---
@tool
def academic_search_tool(queries: List[str]):
    """Search across Semantic Scholar and Exa for academic papers."""
    results = []
    exa_key = os.environ.get("EXA_API_KEY")
    ss_key = os.environ.get("SEMANTIC_SCHOLAR_API_KEY")

    for q in queries:
        try:
            ss_res = requests.get(f"https://api.semanticscholar.org/graph/v1/paper/search?query={q}&limit=2&fields=title,authors,abstract,year", headers={"x-api-key": ss_key} if ss_key else {}).json()
            for p in ss_res.get('data', []):
                authors = ", ".join([a['name'] for a in p.get('authors', [])])
                results.append(f"[Scholar] Title: {p['title']} | Authors: {authors} | Year: {p.get('year')} | Abstract: {p.get('abstract','')[:300]}")
        except: pass

        try:
            exa_res = requests.post("https://api.exa.ai/search", headers={"x-api-key": exa_key}, json={"query": q, "numResults": 2, "useAutoprompt": True}).json()
            for r in exa_res.get('results', []):
                results.append(f"[Exa] Title: {r['title']} | URL: {r['url']}")
        except: pass

    return "\n\n".join(results) if results else "No specific papers found."

class AgentState(TypedDict):
    topic: str
    search_queries: List[str]
    context: str
    report: str
    approved: bool

def reasoning_node(state: AgentState):
    prompt = f"Given the topic '{state['topic']}', suggest 9 specific academic search queries to find recent research gaps."
    res = reasoning_llm.invoke(prompt)
    queries = [line.strip() for line in res.content.split('\n') if '?' not in line and len(line) > 10][:3]
    return {"search_queries": queries}

def hitl_node(state: AgentState):
    decision = interrupt({"question": "Approve these queries?", "queries": state['search_queries']})
    return {"approved": decision.get("approved", False)}

def tool_node(state: AgentState):
    if state.get("approved"):
        data = academic_search_tool.invoke({"queries": state['search_queries']})
        return {"context": data}
    return {"context": "Search was skipped."}

def writer_node(state: AgentState):
    prompt = f"""Act as an academic writer. When writing text, include inline citations using strict APA 7th edition format.
Follow these rules for in-text citations:
1 author: (Author, Year) or Author (Year)
2 authors: (Author 1 & Author 2, Year) or Author 1 and Author 2 (Year)
3+ authors: (Author 1 et al., Year) or Author 1 et al. (Year)
Direct quotes: Always add the page number like (Author, Year, p. 15).
Multiple sources: Separate them with semicolons inside the same brackets, ordered alphabetically, like (Alpha, 2020; Beta, 2022).
Always match every inline citation to a complete reference list entry at the end.

Topic: {state['topic']}
Research Data: {state['context']}
Write a PhD-level literature review synthesis."""
    res = writer_llm.invoke(prompt)
    return {"report": res.content}

builder = StateGraph(AgentState)
builder.add_node("reasoner", reasoning_node)
builder.add_node("approval", hitl_node)
builder.add_node("tools", tool_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "reasoner")
builder.add_edge("reasoner", "approval")
builder.add_edge("approval", "tools")
builder.add_edge("tools", "writer")
builder.add_edge("writer", END)

app = builder.compile(checkpointer=MemorySaver())
print("✅ Graph updated with APA 7 rules.")

In [ ]:
import gradio as gr
import uuid
import tempfile
import os

def start_research(topic):
    if not topic:
        return gr.update(visible=True), gr.update(visible=False), "Please enter a topic.", None, "", gr.update(visible=False), ""

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    app.invoke({"topic": topic, "search_queries": [], "context": "", "report": "", "approved": False}, config)

    state = app.get_state(config)
    if state.next:
        queries = "\n".join([f"- {q}" for q in state.values.get('search_queries', [])])
        status = f"### ⏸ Approval Required\nProposed Queries:\n{queries}"
        return gr.update(visible=False), gr.update(visible=True), status, thread_id, "", gr.update(visible=False), ""

    report = state.values.get("report", "")
    file_path = create_file(report)
    return gr.update(visible=True), gr.update(visible=False), "✅ Completed", thread_id, report, gr.update(visible=True, value=file_path), report

def create_file(content):
    fd, path = tempfile.mkstemp(suffix=".md")
    with os.fdopen(fd, 'w') as tmp:
        tmp.write(content)
    return path

def handle_approval(choice, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    res = app.invoke(Command(resume={"approved": (choice == "Approve")}), config)
    report = res.get("report", "")
    file_path = create_file(report)
    return gr.update(visible=False), "✅ Literature Review Generated", report, gr.update(visible=True, value=file_path), report

def terminate_and_stop():
    return gr.update(visible=True), gr.update(visible=False), "Research Session Closed.", "", gr.update(visible=False), ""

custom_css = """
.gradio-container { max-width: 1000px !important; }
#report-area { min-height: 400px; }
.generating { min-height: 200px; }
"""

with gr.Blocks() as demo:
    gr.Markdown("# 🎓 PhD Research Assistant\n*Powered by Nemotron-3 Ultra (APA 7 Edition)*")
    tid_store = gr.State()

    with gr.Row() as input_row:
        with gr.Column(scale=4):
            topic_input = gr.Textbox(label="Research Topic", placeholder="Enter topic...", lines=2)
        with gr.Column(scale=1):
            run_btn = gr.Button("🚀 Start Research", variant="primary")
            stop_btn = gr.Button("⏹ Close Session", variant="stop", visible=False)

    status_log = gr.Markdown("**Status**: Ready")

    with gr.Group(visible=False) as approval_box:
        gr.Markdown("### 🛠 Search Strategy Approval")
        with gr.Row():
            approve_btn = gr.Button("✅ Approve & Search", variant="success")
            skip_btn = gr.Button("⏭ Skip to Draft", variant="secondary")

    with gr.Column(elem_id="report-area"):
        report_out = gr.Markdown(label="Final Report Output")
        with gr.Row():
            download_file = gr.File(label="Download Report", visible=False)
            copy_btn = gr.Button("📋 Copy to Clipboard", visible=True)

    report_text = gr.Textbox(visible=False)

    copy_btn.click(None, inputs=[report_text], js="(text) => { if(text) { navigator.clipboard.writeText(text); alert('Copied!'); } }")

    run_btn.click(lambda: (gr.update(visible=False), gr.update(visible=True)), None, [run_btn, stop_btn]) \
           .then(start_research, inputs=[topic_input],
                  outputs=[run_btn, approval_box, status_log, tid_store, report_out, download_file, report_text])

    approve_btn.click(lambda tid: handle_approval("Approve", tid), inputs=[tid_store],
                      outputs=[approval_box, status_log, report_out, download_file, report_text])

    skip_btn.click(lambda tid: handle_approval("Skip", tid), inputs=[tid_store],
                   outputs=[approval_box, status_log, report_out, download_file, report_text])

    stop_btn.click(terminate_and_stop, outputs=[run_btn, stop_btn, status_log, report_out, download_file, report_text])

demo.launch(debug=False, share=True, css=custom_css, theme=gr.themes.Soft())